# NB09 — Musanga cover versus deforestation age on native disturbance grids
V5 · CPU · Run 00–03 sequentially. Replaces the previous 50 m exploratory notebook (retained in Git history at bac1092ab241bb4c2f4e9e89083995ac0e520c57). No HLS extraction, model training or NB08 dependency. Uses NB07 final binary canopy maps, the original ROI eroded 100 m, Hansen v1.13 and JRC TMF v2025. Each dataset retains its own actual native CRS, pixel dimensions and lattice origin, checked from Earth Engine source bands (not mosaic default projection). UAV cover estimated by dense 30×30 subpixel sampling per native pixel (~1 m); ≥90% valid support inside ROI. Overlapping mosaics: select highest support, deterministic name tie-break; no double counting. Dates strictly before UAV survey calendar year. No detected event is not age zero or proof of undisturbed forest. JRC deforestation age = elapsed time since latest transition from mapped forest (1,2,4) to deforested land (3); persistent class 3 does not reset the clock. Regrowth age = elapsed time since onset of current continuous class 4 episode following mapped deforestation (3). Other transitions and left-censored histories are reported, not assigned invented ages. Retrospective classifications can use later imagery. Spatial chronosequence, not longitudinal succession or causal evidence.
Sources: https://developers.google.com/earth-engine/datasets/catalog/UMD_hansen_global_forest_change_2025_v1_13 ; https://forobs.jrc.ec.europa.eu/TMF/resources/tutorial/gee .

**Extension:** after completing 00–03, run 04 for native-grid annual age summaries and latest mapped degradation onset; 05 for a separate 50 m HLS spectral-level/dynamics ablation using existing matched validation data. Fresh runtime after a successful native run: 00 → 01 → 04 → 05 (skip 02–03). No UAV reaggregation or HLS download. 04 requires a small repeat JRC-only history extraction because the native tables did not store degradation dates. Original native tables and outputs stay unchanged.


**Native HLS ecological extension (06):** in a fresh runtime run 00 → 06A → 06B → 06C. No need to rerun 01–05. New HLS annual extraction and UAV aggregation on the actual HLS native 30 m lattice; resumable tile caches. Block 05 remains a separate matched 50 m predictive ablation, not used for native-grid ecology. All earlier outputs are preserved.


In [ ]:
#@title 00 — Mount Drive, verify UAV inputs and study region
%pip -q install earthengine-api rasterio geopandas pyarrow scipy pandas matplotlib
from pathlib import Path
import os,json,hashlib,math,time
from collections import defaultdict
from contextlib import ExitStack
import numpy as np,pandas as pd,geopandas as gpd,matplotlib.pyplot as plt
import rasterio,ee
from rasterio import Affine
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.ops import unary_union
from scipy.stats import spearmanr
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/V5'),Path('/content/drive/MyDrive/Musanga/Final/V5')] if (p.parent/'Data').is_dir() and p.is_dir()),None)
assert ROOT is not None
SITES=['Baego','MOS','Yangambi','Yoko'];YEARS=dict(Baego=2023,MOS=2024,Yangambi=2024,Yoko=2020)
PROJECT='musanga-511110';CRS='EPSG:32635';SUPPORT=.90;SUB=30;TILE=64
GID='UMD/hansen/global_forest_change_2025_v1_13';JID='projects/JRC/TMF/v1_2025/AnnualChanges'
FORCE_REBUILD=False
def digest(x):return hashlib.sha256(json.dumps(x,sort_keys=True,default=str).encode()).hexdigest()
def atomic_json(p,x):
 tmp=p.with_suffix('.partial');tmp.write_text(json.dumps(x,indent=2,default=str));os.replace(tmp,p)
def atomic_parquet(p,x):
 tmp=p.with_suffix('.partial');x.to_parquet(tmp,index=False);os.replace(tmp,p)
NB07=ROOT/'Outputs/07_RGB_unified_production_v5';M=json.loads((NB07/'07_production_mosaic_manifest.json').read_text())
assert M['complete'] and M['schema']=='musanga_v5_nb07_rgb_mosaics' and len(M['mosaics'])==6
MOS=defaultdict(list);meta=[]
for item in M['mosaics']:
 p=NB07/'reconstructed_mosaics'/Path(item['output']).name
 assert p.is_file(),f'Missing NB07 mosaic: {p}'
 rec=json.loads((p.parent/(item['name']+'_reconstruction.json')).read_text())
 assert rec['status']=='complete' and rec['digest']==item['digest'] and rec['model']==M['model_sha'] and rec['policy']==M['policy_sha']
 with rasterio.open(p) as ds:
  assert ds.count==2 and ds.nodata==-9999 and ds.crs==rasterio.crs.CRS.from_string(CRS)
  assert ds.tags().get('signature')==item['digest'] and ds.width==item['width'] and ds.height==item['height']
  assert ds.descriptions==('Musanga probability','Musanga binary at 0.5')
  b=list(ds.bounds)
 MOS[item['site']].append((item['name'],p,b));meta.append(dict(name=item['name'],site=item['site'],signature=item['digest'],size=p.stat().st_size,bounds=b))
assert set(MOS)==set(SITES)
for s in SITES:MOS[s].sort(key=lambda x:x[0])
raw=json.loads((ROOT/'Outputs/00_input_audit/00_input_manifest.json').read_text())
rp=Path(raw['roi'])
if not rp.is_file():rp=ROOT.parent/'Data/Ortho_RGB'/rp.name
assert rp.is_file()
roi=unary_union(gpd.read_file(rp).to_crs(CRS).geometry).buffer(-100.)
assert roi.is_valid and not roi.is_empty
GEOM={s:roi.intersection(unary_union([box(*b) for _,_,b in MOS[s]])) for s in SITES}
assert all(not g.is_empty for g in GEOM.values())
try:ee.Initialize(project=PROJECT)
except Exception:ee.Authenticate();ee.Initialize(project=PROJECT)
print('Verified six NB07 mosaics; no new UAV inference.')


In [ ]:
#@title 01 — Obtain and verify the original Hansen and JRC lattices
G=ee.Image(GID);J=ee.ImageCollection(JID)
def projinfo(img,band):
 p=img.select(band).projection().getInfo()
 a=Affine(*p['transform']);assert abs(a.b)<1e-12 and abs(a.d)<1e-12 and a.a>0 and a.e<0,p
 return p['crs'],a
def samegrid(p,q):
 c,a=p;d,b=q
 if c!=d or not np.allclose([a.a,a.b,a.d,a.e],[b.a,b.b,b.d,b.e],rtol=1e-9,atol=1e-12):return False
 x,y=(~a)*(b.c,b.f)
 return abs(x-round(x))<1e-5 and abs(y-round(y))<1e-5
PG=projinfo(G,'lossyear')
for band in ['datamask','treecover2000']:assert samegrid(PG,projinfo(G,band))
GRIDS={};IMAGES={};REGIONS={};TILES={}
for s in SITES:
 y=YEARS[s];g=GEOM[s]
 lonlat=gpd.GeoSeries([g],crs=CRS).to_crs('EPSG:4326').iloc[0]
 region=ee.Geometry.Rectangle(list(lonlat.bounds),proj='EPSG:4326',geodesic=False)
 jc=J.filterBounds(region).sort('system:index');ids=jc.aggregate_array('system:index').getInfo()
 assert ids,f'No JRC source tiles at {s}'
 bands=[f'Dec{k}' for k in range(1990,y)]
 js=jc.toList(len(ids));pj=projinfo(ee.Image(js.get(0)),bands[0])
 # Verify every selected annual source band on every intersecting original tile.
 for i in range(len(ids)):
  src=ee.Image(js.get(i));info=src.select(bands).getInfo()['bands']
  for b in info:
   assert samegrid(pj,(b['crs'],Affine(*b['crs_transform']))),(s,b)
 # Use source projection rather than default one-degree mosaic projection.
 jimg=jc.select(bands).mosaic().setDefaultProjection(pj[0],list(pj[1])[:6]).unmask(255).toUint8()
 land=G.select('datamask');loss=G.select('lossyear').unmask(0)
 gimg=ee.Image.cat([land.unmask(255),loss.updateMask(land.eq(1)).unmask(255),G.select('treecover2000').unmask(255)]).toUint8()
 for tag,p,img in [('Hansen',PG,gimg),('JRC',pj,jimg)]:
  c,a=p;GRIDS[s,tag]=p;IMAGES[s,tag]=img
  geom=gpd.GeoSeries([g],crs=CRS).to_crs(c).iloc[0];REGIONS[s,tag]=geom
  x0,y0,x1,y1=geom.bounds
  cl,rt=(~a)*(x0,y1);cr,rb=(~a)*(x1,y0)
  tiles=[]
  for row in range(math.floor(rt/TILE)*TILE,math.ceil(rb/TILE)*TILE,TILE):
   for col in range(math.floor(cl/TILE)*TILE,math.ceil(cr/TILE)*TILE,TILE):
    l,t=a*(col,row);r,b=a*(col+TILE,row+TILE)
    if geom.intersects(box(l,b,r,t)):tiles.append((row,col))
  assert tiles;TILES[s,tag]=tiles
  print(s,tag,c,list(a)[:6],len(tiles),'tiles')
CFG=dict(version='native_ecology_v1',sources=[GID,JID],years=YEARS,roi_sha=hashlib.sha256(roi.wkb).hexdigest(),mosaics=meta,model=M['model_sha'],policy=M['policy_sha'],support=SUPPORT,sub=SUB,tile=TILE,grids={f'{s}_{t}':[c,list(a)[:6]] for (s,t),(c,a) in GRIDS.items()})
OUT=ROOT/'Outputs/09_ecology_v5'/digest(CFG)[:20];OUT.mkdir(parents=True,exist_ok=True)
CACHE=OUT/'tiles';CACHE.mkdir(exist_ok=True);atomic_json(OUT/'config.json',CFG)
print('Native-grid output:',OUT)


In [ ]:
#@title 02 — Aggregate UAV cover directly to native pixels and attach disturbance histories
def native_tile(s,tag,row,col,stack):
 c,a=GRIDS[s,tag];tr=a*Affine.translation(col,row)
 # Approximate area integration by equally spaced subpixel centres within each native footprint.
 hi=tr*Affine.scale(1/SUB,1/SUB);size=TILE*SUB
 inside=rasterize([(REGIONS[s,tag],1)],out_shape=(size,size),transform=hi,fill=0,dtype='uint8').astype(bool)
 def reduce(z):return z.reshape(TILE,SUB,TILE,SUB).sum((1,3))
 best=np.zeros((TILE,TILE),dtype=np.int32);positive=best.copy();source=np.full(best.shape,-1,dtype=np.int16)
 for i,(name,p,b) in enumerate(MOS[s]):
  ds=stack[name]
  with WarpedVRT(ds,crs=c,transform=hi,width=size,height=size,resampling=Resampling.nearest,src_nodata=-9999,nodata=-9999,warp_mem_limit=128) as vrt:
   z=vrt.read(2)
  valid=inside&((z==0)|(z==1));nv=reduce(valid);npix=reduce(valid&(z==1))
  better=nv>best;best[better]=nv[better];positive[better]=npix[better];source[better]=i
 keep=best>=math.ceil(SUPPORT*SUB*SUB)
 if not keep.any():return pd.DataFrame()
 # Explicit PixelGrid affine; Earth Engine returns structured NPY, not a resampled arbitrary 30m UTM tile.
 grid=dict(dimensions=dict(width=TILE,height=TILE),affineTransform=dict(scaleX=tr.a,shearX=tr.b,translateX=tr.c,shearY=tr.d,scaleY=tr.e,translateY=tr.f),crsCode=c)
 for attempt in range(5):
  try:
   arr=ee.data.computePixels(dict(expression=IMAGES[s,tag],fileFormat='NUMPY_NDARRAY',grid=grid));break
  except Exception:
   if attempt==4:raise
   time.sleep(2**attempt)
 assert arr.shape==(TILE,TILE)
 rr,cc=np.where(keep);xx,yy=tr*(cc+.5,rr+.5)
 d=pd.DataFrame(dict(site=s,product=tag,row=rr+row,col=cc+col,x_native=xx,y_native=yy,survey_year=YEARS[s],support=best[keep]/(SUB*SUB),cover_pp=100*positive[keep]/best[keep],mosaic=[MOS[s][i][0] for i in source[keep]]))
 y=YEARS[s]
 if tag=='Hansen':
  assert set(arr.dtype.names)=={'datamask','lossyear','treecover2000'},arr.dtype.names
  mask=arr['datamask'][keep];loss=arr['lossyear'][keep].astype(int);tc=arr['treecover2000'][keep]
  ok=(mask==1)&(loss>=0)&(loss<=25)
  prior=ok&(loss>0)&(2000+loss<y)
  d['history_valid']=ok;d['loss_year']=np.where(prior,2000+loss,np.nan)
  d['age_since_loss']=y-d.loss_year
  d['loss_status']=np.select([~ok,prior,ok&(loss>0)&(2000+loss>=y)],['invalid','prior loss','survey year or later loss'],default='no detected loss')
  d['treecover2000']=np.where(tc<=100,tc,np.nan)
 else:
  years=np.arange(1990,y);names=[f'Dec{k}' for k in years]
  assert set(arr.dtype.names)==set(names),arr.dtype.names
  hist=np.stack([arr[b][keep] for b in names],axis=0)
  ok=np.isin(hist,np.arange(1,7)).all(axis=0)
  # Only genuine transitions from forest to class 3, not each year spent deforested.
  entries=np.isin(hist[:-1],[1,2,4])&(hist[1:]==3)
  latest=np.max(np.where(entries,years[1:,None],0),axis=0)
  # Current uninterrupted class 4 episode, with directly preceding deforested state.
  run=np.cumprod((hist[::-1]==4).astype(np.int16),axis=0).sum(axis=0)
  onset=len(years)-run;prev=hist[np.maximum(onset-1,0),np.arange(len(d))]
  confirmed=ok&(run>0)&(onset>0)&(prev==3)
  d['history_valid']=ok;d['jrc_state']=np.where(ok,hist[-1],np.nan)
  d['deforestation_year']=np.where(ok&(latest>0),latest,np.nan)
  d['age_since_deforestation']=y-d.deforestation_year
  d['regrowth_onset_year']=np.where(confirmed,years[0]+onset,np.nan)
  d['age_since_regrowth']=y-d.regrowth_onset_year
  d['deforestation_status']=np.select([~ok,latest>0,(hist==3).any(axis=0)],['invalid','dated forest-to-deforested transition','deforested state without dated forest transition'],default='no detected deforestation')
  d['regrowth_status']=np.select([~ok,confirmed,run>0],['invalid','dated post-deforestation regrowth','other or left-censored regrowth'],default='not currently regrowth')
 return d
all_tables={}
for s in SITES:
 with ExitStack() as es:
  stack={name:es.enter_context(rasterio.open(p)) for name,p,b in MOS[s]}
  for tag in ['Hansen','JRC']:
   chunks=[]
   for i,(row,col) in enumerate(TILES[s,tag]):
    p=CACHE/f'{s}_{tag}_{row}_{col}.parquet'
    if p.is_file() and not FORCE_REBUILD:d=pd.read_parquet(p)
    else:
     d=native_tile(s,tag,row,col,stack);atomic_parquet(p,d)
    if len(d):chunks.append(d)
    print(s,tag,i+1,'/',len(TILES[s,tag]),'pixels',len(d),flush=True)
   assert chunks,(s,tag)
   all_tables[s,tag]=pd.concat(chunks,ignore_index=True)
for tag in ['Hansen','JRC']:
 d=pd.concat([all_tables[s,tag] for s in SITES],ignore_index=True)
 assert not d.duplicated(['site','row','col']).any()
 assert d.cover_pp.between(0,100).all() and d.support.ge(SUPPORT).all()
 atomic_parquet(OUT/f'{tag}_native_pixels.parquet',d)
 display(d.groupby('site').agg(n=('cover_pp','size'),valid_history=('history_valid','sum'),mean_support=('support','mean')))


In [ ]:
#@title 03 — Four simple figures: raw pixel-level cover against age and JRC state
H=pd.read_parquet(OUT/'Hansen_native_pixels.parquet');J9=pd.read_parquet(OUT/'JRC_native_pixels.parquet')
rows=[]
for d,col,label in [(H,'age_since_loss','Years since Hansen detected tree-cover loss'),(J9,'age_since_deforestation','Years since JRC forest-to-deforested transition'),(J9,'age_since_regrowth','Years since onset of JRC post-deforestation regrowth')]:
 fig,axs=plt.subplots(2,2,figsize=(11,8),sharex=True,sharey=True,layout='constrained')
 maxage=max(1,d[col].max()) if d[col].notna().any() else 1
 for s,ax in zip(SITES,axs.ravel()):
  g=d[d.site.eq(s)];q=g[g.history_valid&g[col].notna()]
  x=q[col];y=q.cover_pp
  rho=spearmanr(x,y).statistic if len(q)>2 and x.nunique()>1 and y.nunique()>1 else np.nan
  ax.scatter(x,y,s=5,alpha=.18,rasterized=True)
  ax.set(title=f'{s} | n={len(q):,}',xlabel=label,ylabel='UAV Musanga cover (%)',xlim=(0,maxage+1),ylim=(0,100))
  rows.append(dict(site=s,variable=col,n=len(q),n_total=len(g),n_valid_history=int(g.history_valid.sum()),rho=rho))
 fig.savefig(OUT/f'{col}.png',dpi=220);plt.show();plt.close(fig)
names={1:'Undisturbed TMF',2:'Degraded TMF',3:'Deforested land',4:'Regrowth',5:'Water',6:'Other land cover'}
fig,axs=plt.subplots(2,2,figsize=(12,8),sharey=True,layout='constrained')
for s,ax in zip(SITES,axs.ravel()):
 g=J9[J9.site.eq(s)&J9.history_valid];groups=[];labels=[]
 for k,v in names.items():
  q=g[g.jrc_state.eq(k)].cover_pp
  if len(q):groups.append(q.to_numpy());labels.append(f'{v}\n(n={len(q):,})')
 if groups:ax.boxplot(groups,tick_labels=labels,showfliers=True)
 ax.tick_params(axis='x',labelrotation=30,labelsize=8);ax.set(title=s,ylabel='UAV Musanga cover (%)',ylim=(0,100))
fig.savefig(OUT/'JRC_state_cover.png',dpi=220);plt.show();plt.close(fig)
summary=pd.DataFrame(rows);summary.to_csv(OUT/'age_scatter_summary.csv',index=False);display(summary)
for d,col in [(H,'loss_status'),(J9,'deforestation_status'),(J9,'regrowth_status'),(J9,'jrc_state')]:
 q=d.groupby(['site',col],dropna=False).cover_pp.agg(n='size',mean_cover='mean',median_cover='median').reset_index()
 q.to_csv(OUT/f'{col}_summary.csv',index=False);display(q.round(2))
atomic_json(OUT/'completion.json',dict(complete=True,config=CFG,n_Hansen=len(H),n_JRC=len(J9),outputs=sorted(p.name for p in OUT.iterdir())))
print('NB09 NATIVE GRID COMPLETE',OUT)


## 04 — Annual age distributions and time since degradation
One point per integer age and forest: mean, median, interquartile range, n and number of 500 m spatial blocks. All dated pixels retained; sparse ages explicitly marked. No smoothing or fitted curve. IQR describes variability, not a confidence interval. JRC degradation age is elapsed time since latest transition from mapped undisturbed or regrowing forest (1 or 4) into class 2; persistent class 2 does not reset age. Re-entry after a nonforest state is not treated as confirmed forest degradation. Pixels starting in class 2 are left-censored unless a later dated entry occurs. Re-read JRC source histories on the exact native grid; re-use stored UAV fractions and verify agreement with previous latest-state and deforestation-year fields. No UAV inference/aggregation.

In [ ]:
#@title 04 — Native-grid annual cover statistics and latest JRC degradation age
H=pd.read_parquet(OUT/'Hansen_native_pixels.parquet');J9=pd.read_parquet(OUT/'JRC_native_pixels.parquet')
assert (OUT/'completion.json').is_file() and json.loads((OUT/'completion.json').read_text())['complete']
O4=OUT/'04_annual_ages_and_degradation_v1';O4.mkdir(exist_ok=True)
DGC=O4/'jrc_tiles';DGC.mkdir(exist_ok=True)
def degradation_tile(s,row,col,g):
 c,a=GRIDS[s,'JRC'];tr=a*Affine.translation(col,row)
 grid=dict(dimensions=dict(width=TILE,height=TILE),affineTransform=dict(scaleX=tr.a,shearX=tr.b,translateX=tr.c,shearY=tr.d,scaleY=tr.e,translateY=tr.f),crsCode=c)
 for attempt in range(5):
  try:arr=ee.data.computePixels(dict(expression=IMAGES[s,'JRC'],fileFormat='NUMPY_NDARRAY',grid=grid));break
  except Exception:
   if attempt==4:raise
   time.sleep(2**attempt)
 years=np.arange(1990,YEARS[s]);bands=[f'Dec{k}' for k in years]
 assert arr.shape==(TILE,TILE) and set(arr.dtype.names)==set(bands)
 rr=g.row.to_numpy(int)-row;cc=g.col.to_numpy(int)-col
 hist=np.stack([arr[b][rr,cc] for b in bands])
 ok=np.isin(hist,np.arange(1,7)).all(axis=0)
 assert np.array_equal(ok,g.history_valid.to_numpy())
 assert np.array_equal(np.where(ok,hist[-1],np.nan),g.jrc_state.to_numpy(),equal_nan=True)
 defo=np.max(np.where(np.isin(hist[:-1],[1,2,4])&(hist[1:]==3),years[1:,None],0),axis=0)
 assert np.array_equal(np.where(ok&(defo>0),defo,np.nan),g.deforestation_year.to_numpy(),equal_nan=True)
 entry=np.isin(hist[:-1],[1,4])&(hist[1:]==2)
 latest=np.max(np.where(entry,years[1:,None],0),axis=0)
 d=g[['site','row','col']].copy()
 d['degradation_year']=np.where(ok&(latest>0),latest,np.nan)
 d['age_since_degradation']=YEARS[s]-d.degradation_year
 d['n_degradation_entries']=np.where(ok,entry.sum(axis=0),np.nan)
 d['degradation_status']=np.select([~ok,latest>0,(hist==2).any(axis=0)],['invalid','dated forest-to-degraded transition','degraded state without dated forest transition'],default='no detected degradation')
 return d
parts=[]
for (s,row,col),g in J9.assign(tile_row=(J9.row//TILE)*TILE,tile_col=(J9.col//TILE)*TILE).groupby(['site','tile_row','tile_col']):
 row=int(row);col=int(col);p=DGC/f'{s}_{row}_{col}.parquet'
 if p.is_file() and not FORCE_REBUILD:q=pd.read_parquet(p)
 else:q=degradation_tile(s,row,col,g);atomic_parquet(p,q)
 assert q[['site','row','col']].reset_index(drop=True).equals(g[['site','row','col']].reset_index(drop=True))
 parts.append(q);print(s,row,col,'JRC pixels',len(q),flush=True)
DJ=J9.merge(pd.concat(parts,ignore_index=True),on=['site','row','col'],validate='one_to_one')
assert len(DJ)==len(J9)
atomic_parquet(O4/'JRC_native_degradation.parquet',DJ)
def add_blocks(d,tag):
 q=d.copy();q['block500']=''
 for s,g in q.groupby('site'):
  xy=gpd.GeoSeries(gpd.points_from_xy(g.x_native,g.y_native),index=g.index,crs=GRIDS[s,tag][0]).to_crs(CRS)
  bx=np.floor(xy.x.to_numpy()/500).astype(int);by=np.floor(xy.y.to_numpy()/500).astype(int)
  q.loc[g.index,'block500']=[f'{x}_{y}' for x,y in zip(bx,by)]
 return q
HH=add_blocks(H,'Hansen');JJ=add_blocks(DJ,'JRC')
specs=[(HH,'age_since_loss','Hansen detected loss'),(JJ,'age_since_deforestation','JRC deforestation'),(JJ,'age_since_regrowth','JRC regrowth onset'),(JJ,'age_since_degradation','JRC degradation onset')]
tables=[]
for d,col,label in specs:
 q=d[d.history_valid&d[col].notna()].copy()
 assert (q[col]>0).all() and np.allclose(q[col],q[col].astype(int))
 q['age']=q[col].astype(int)
 a=q.groupby(['site','age']).agg(n=('cover_pp','size'),blocks=('block500','nunique'),mean=('cover_pp','mean'),median=('cover_pp','median'),q25=('cover_pp',lambda z:z.quantile(.25)),q75=('cover_pp',lambda z:z.quantile(.75))).reset_index()
 a['variable']=col;a['sufficient_support']=(a.n>=30)&(a.blocks>=5);tables.append(a)
 maxage=max(1,q.age.max()) if len(q) else 1
 fig,axs=plt.subplots(2,2,figsize=(12,8),sharex=True,sharey=True,layout='constrained')
 for s,ax in zip(SITES,axs.ravel()):
  g=q[q.site.eq(s)];z=a[a.site.eq(s)]
  ax.scatter(g.age,g.cover_pp,s=3,alpha=.07,color='grey',rasterized=True)
  for robust,zz in z.groupby('sufficient_support'):
   ax.errorbar(zz.age,zz['median'],yerr=[zz['median']-zz.q25,zz.q75-zz['median']],fmt='o' if robust else 'o',mfc='C0' if robust else 'none',mec='C0',ecolor='C0',alpha=1 if robust else .45,label='Median / IQR' if robust else 'Sparse age: median / IQR')
  if len(z):ax.scatter(z.age,z['mean'],marker='x',s=25,color='C1',label='Mean')
  ax.set(title=f'{s} | n={len(g):,}',xlabel='Years since '+label,ylabel='UAV Musanga cover (%)',xlim=(0,maxage+1),ylim=(0,100))
  if len(z):ax.legend(fontsize=7)
 fig.savefig(O4/f'{col}_annual.png',dpi=200);plt.show();plt.close(fig)
summary=pd.concat(tables,ignore_index=True);summary.to_csv(O4/'annual_age_summary.csv',index=False);display(summary.round(2))
q=DJ.groupby(['site','degradation_status'],dropna=False).cover_pp.agg(n='size',mean='mean',median='median').reset_index()
q.to_csv(O4/'degradation_status_summary.csv',index=False);display(q.round(2))
# Strictly descriptive sensitivity: degradation-only history (no mapped deforested state ever)
# is not inferred from absent dated transition; current analysis reports all dated degradation events.
atomic_json(O4/'completion.json',dict(complete=True,n=len(DJ),source=str(OUT),definition='latest observed 1/4 to 2 transition; annual IQR not CI'))
print('04 COMPLETE',O4)


## 05 — Separate 50 m HLS spectral-level versus dynamics ablation
This is a methodological sensitivity test, not the native-grid ecological chronosequence. Reuses exact matched NB08bis Block 14 50 m reference, same spatial folds, 100 m exclusion, LOFO and equal-forest HGB settings. Test whether six long-term medians of annual spectral observations explain the HLS benefit, versus recent-three-year medians, variability/change descriptors, both, and the full HLS set. Include baseline S2+EMB and baseline+long-term medians / baseline+full HLS. Full available observation windows differ by survey year/site; this is a limitation of the retrospective comparison. No new HLS or UAV extraction; caches kept separate. Paired block bootstrap conditional on fixed predictions and four forests, not regional inference.

In [ ]:
#@title 05 — HLS spectral medians versus temporal dynamics (cached matched 50 m)
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn import __version__ as SK_VERSION
from scipy.spatial import cKDTree
from threadpoolctl import threadpool_limits
from sklearn.metrics import roc_auc_score
threadpool_limits(limits=1)
def dataframe_sha(d):return hashlib.sha256(pd.util.hash_pandas_object(d,index=False).values.tobytes()).hexdigest()
def load_hls_cohort():
 ANNUAL=[f'RAW_ANNUAL__{i:02d}' for i in range(1,16)]
 QUARTER=[f'S2_QUARTERLY_3YR__{i:02d}' for i in range(1,45)]
 EMB=[f'EMB_ANNUAL__{i:02d}' for i in range(1,65)]
 Q_COUNTS=[QUARTER[i-1] for i in (11,22,33,44)]
 S2=ANNUAL+[c for c in QUARTER if c not in Q_COUNTS]
 FEATURE_SETS={'M1_S2':S2,'M2_EMB':EMB,'M3_S2_EMB':S2+EMB}
 ALL_FEATURES=sorted(set(sum(FEATURE_SETS.values(),[])))
 BASE=S2+EMB;BASE_NAME='M3_S2_EMB__HGB__forest_equal'
 KEYS=['site','mosaic','row','col'];MIN_SUPPORT=.90
 SATROOT=ROOT/'Outputs/08_satellite_reference';NB08_OUT=ROOT/'Outputs/08_satellite_science_v5';EXPL=ROOT/'Outputs/08bis_abundance_exploration_v5'
 candidates=[]
 for p in sorted(SATROOT.glob('native_50m_reference_*.parquet')):
  sig=p.stem.removeprefix('native_50m_reference_');cf=NB08_OUT/sig/'config.json'
  if not cf.is_file():continue
  c=json.loads(cf.read_text());tc=SATROOT/'uav_targets'/c['target_id']/'config.json'
  if not tc.is_file() or digest(c)[:20]!=sig or c['features']!=FEATURE_SETS:continue
  t=json.loads(tc.read_text())
  if t['model']!=M['model_sha'] or t['policy']!=M['policy_sha'] or t['support']!=MIN_SUPPORT:continue
  old={a['name']:a for a in t['mosaics']}
  if set(old)!={a['name'] for a in M['mosaics']}:continue
  valid=True
  for a in M['mosaics']:
   f=NB07/'reconstructed_mosaics'/Path(a['output']).name
   if not f.is_file() or old[a['name']]['digest']!=a['digest'] or old[a['name']]['size']!=f.stat().st_size:valid=False;break
   with rasterio.open(f) as ds:
    if ds.tags().get('signature')!=a['digest'] or list(ds.bounds)!=old[a['name']]['bounds']:valid=False;break
  if not valid:continue
  d=pd.read_parquet(p)
  if dataframe_sha(d[KEYS+['x','y','target_fraction']+ALL_FEATURES])==c['source_sha256']:candidates.append((p,c,d))
 assert candidates,'Complete NB08 reference first'
 assert len({c['source_sha256'] for _,c,_ in candidates})==1,'Multiple distinct references; select explicitly'
 REF,CONFIG,D=candidates[0];SOURCE_SHA=CONFIG['source_sha256']
 SAT_ID=CONFIG['satellite_id'];SAT_DIR=SATROOT/'satellite_10m'/SAT_ID
 TARGET_DIR=SATROOT/'uav_targets'/CONFIG['target_id']
 assert len(D)==CONFIG['n'] and set(D.site)==set(SITES) and not D.duplicated(['site','row','col']).any()
 assert D.target_fraction.between(0,1).all() and np.isfinite(D[BASE].to_numpy(float)).all()
 found=[]
 for p in sorted((NB08_OUT/'within_landscape_50m').glob('*/heldout_predictions.parquet')):
  c=json.loads((p.parent/'config.json').read_text());z=pd.read_parquet(p)
  if c.get('version')!='within_landscape_v2' or c.get('buffer')!=100 or c.get('block')!=500 or BASE_NAME not in z:continue
  cols=['site','row','col','x','y','target_fraction']
  if not z[cols].equals(D[cols]):continue
  used=sorted(set(sum(c['features'].values(),[])))
  if dataframe_sha(D.assign(fold21=z.fold21.to_numpy())[['site','x','y','target_fraction','fold21']+used])==c['source']:found.append((p,z))
 assert found,'Complete NB08 spatial holdout first'
 BASE_P,OLD=found[0];M21=D.assign(block21=OLD.block21.to_numpy(),fold21=OLD.fold21.to_numpy())
 
 foldsha=dataframe_sha(M21[['site','row','col','fold21']])
 candidates=[]
 for p in EXPL.glob('*/14_long_history/*/hansen_mask_corrected_v2/features_50m.parquet'):
  c=json.loads((p.parent/'config.json').read_text())
  if c['parent']['source']==SOURCE_SHA and c['parent']['fold']==foldsha:candidates.append(p)
 assert len(candidates)==1,candidates
 F=pd.read_parquet(candidates[0])
 assert len(F)==len(M21) and not F.duplicated(KEYS).any()
 d=M21.merge(F,on=KEYS,validate='one_to_one')
 assert len(d)==len(M21)
 return d[d.eligible14].reset_index(drop=True),BASE,str(candidates[0])
 
d5,BASE5,source5=load_hls_cohort()
assert set(d5.site)==set(SITES) and d5.groupby('site').fold21.nunique().eq(5).all()
h=[c for c in d5 if c.startswith('H14_')]
level=[c for c in h if c.endswith('_median')]
recent=[c for c in h if c.endswith('_recent3')]
dyn=[c for c in h if c.endswith(('_sd','_range','_trend','_largest_drop','_drop_age','_decline_fraction','_recovery_slope'))]
ext=[c for c in h if c.endswith(('_min','_max'))]
assert len(level)==len(recent)==6 and set(h)==set(level+recent+dyn+ext),(level,recent,set(h)-set(level+recent+dyn+ext))
# Dynamics include variability summaries; not all features are time-order dependent.
arms={'baseline':BASE5,'longterm_median':level,'recent3_median':recent,'dynamics':dyn,'median_dynamics':level+dyn,'full_HLS':h,'baseline_plus_median':BASE5+level,'baseline_plus_full_HLS':BASE5+h}
model=dict(learning_rate=.06,max_iter=180,max_leaf_nodes=20,min_samples_leaf=35,l2_regularization=2,early_stopping=False)
seed=20261123;cols=['site','row','col','x','y','target_fraction','block21','fold21']
cfg=dict(version='HLS_level_dynamics_v1',source=source5,sha=dataframe_sha(d5[cols+list(dict.fromkeys(BASE5+h))]),arms=arms,model=model,seed=seed,sklearn=SK_VERSION,buffer=100)
O5=ROOT/'Outputs/09_ecology_v5'/'05_HLS_ablation'/digest(cfg)[:20];O5.mkdir(parents=True,exist_ok=True);atomic_json(O5/'config.json',cfg)
def splits5():
 xy=d5[['x','y']].to_numpy();site=d5.site.to_numpy();fold=d5.fold21.to_numpy()
 for k in range(5):
  te=np.flatnonzero(fold==k);tr=np.flatnonzero(fold!=k);keep=np.ones(len(tr),bool)
  for s in SITES:
   a=np.flatnonzero(site[tr]==s);b=te[site[te]==s]
   if len(a) and len(b):keep[a]=cKDTree(xy[b]).query(xy[tr[a]])[0]>=100
  assert len(te) and keep.any()
  yield k,tr[keep],te
pred={}
for protocol in ['spatial','LOFO']:
 p=d5[cols].copy()
 sp=list(splits5()) if protocol=='spatial' else [(s,np.flatnonzero(d5.site.ne(s)),np.flatnonzero(d5.site.eq(s))) for s in SITES]
 for k,tr,te in sp:
  g=d5.iloc[tr];sw=1/g.groupby('site').site.transform('size').to_numpy(float);sw/=sw.mean()
  for name,features in arms.items():
   f=O5/f'{protocol}_{k}_{name}.npz'
   if f.is_file() and not FORCE_REBUILD:
    with np.load(f,allow_pickle=False) as z:assert np.array_equal(z['test'],te);v=z['pred']
   else:
    m=HistGradientBoostingRegressor(random_state=seed,**model);m.fit(g[features].to_numpy(np.float32),g.target_fraction.to_numpy(),sample_weight=sw)
    v=np.clip(m.predict(d5.iloc[te][features].to_numpy(np.float32)),0,1)
    tmp=f.with_suffix('.partial')
    with open(tmp,'wb') as z:np.savez_compressed(z,test=te,pred=v)
    os.replace(tmp,f)
   assert len(v)==len(te) and np.isfinite(v).all()
   p.loc[te,name]=v;print(protocol,k,name,flush=True)
 assert p[list(arms)].notna().all().all()
 atomic_parquet(O5/f'{protocol}_predictions.parquet',p);pred[protocol]=p
def metrics5(y,z,w):
 y=np.asarray(y);z=np.asarray(z);w=np.asarray(w,float);w/=w.sum()
 e=z-y;var=np.sum(w*(y-np.sum(w*y))**2)
 h=y>=.4
 return dict(R2=1-np.sum(w*e**2)/var if var>0 else np.nan,MAE_pp=100*np.sum(w*abs(e)),RMSE_pp=100*np.sqrt(np.sum(w*e**2)),bias_pp=100*np.sum(w*e),AUC40=roc_auc_score(h,z,sample_weight=w) if h.any() and (~h).any() else np.nan)
summary=[];per=[];boot=[]
pairs=[('longterm_median','recent3_median'),('dynamics','longterm_median'),('median_dynamics','longterm_median'),('full_HLS','longterm_median'),('full_HLS','median_dynamics'),('baseline_plus_median','baseline'),('baseline_plus_full_HLS','baseline_plus_median')]
for protocol,p in pred.items():
 w=1/p.groupby('site').site.transform('size').to_numpy(float);w/=w.sum()
 den=np.mean([g.target_fraction.var(ddof=0) for _,g in p.groupby('site')])
 for name in arms:
  m=metrics5(p.target_fraction,p[name],w.copy())
  auc=[]
  for s,g in p.groupby('site'):
   b=metrics5(g.target_fraction,g[name],np.ones(len(g)));auc.append(b['AUC40']);per.append(dict(protocol=protocol,model=name,site=s,n=len(g),**b))
  summary.append(dict(protocol=protocol,model=name,n=len(p),**m,within_forest_R2=1-(m['RMSE_pp']/100)**2/den,mean_site_AUC40=np.nanmean(auc)))
 for name,ref in pairs:
  delta=100*(abs(p[name]-p.target_fraction)-abs(p[ref]-p.target_fraction))
  rng=np.random.default_rng(seed);dist=[]
  for s,g in p.assign(delta=delta).groupby('site'):
   q=g.groupby('block21').delta.agg(['sum','count']).to_numpy()
   v=q[rng.integers(0,len(q),(1000,len(q)))].sum(1);dist.append(v[:,0]/v[:,1])
  v=np.mean(dist,axis=0)
  boot.append(dict(protocol=protocol,model=name,reference=ref,delta_MAE_pp=np.sum(w*delta),ci_low=np.quantile(v,.025),ci_high=np.quantile(v,.975)))
S=pd.DataFrame(summary);P=pd.DataFrame(per);B=pd.DataFrame(boot)
S.to_csv(O5/'summary.csv',index=False);P.to_csv(O5/'by_forest.csv',index=False);B.to_csv(O5/'paired_bootstrap.csv',index=False)
display(S.round(3));display(B.round(3))
atomic_json(O5/'completion.json',dict(complete=True,n=len(d5),config=cfg))
print('05 COMPLETE',O5)


## 06 — Native HLS 30 m ecology: observed spectral history versus UAV Musanga fraction

HLS L30 and S30 are harmonized 30 m observations. Use the native projection and affine transform from source imagery; inventory the source lattices across both collections and every pre-survey year. Choose one compatible native lattice per forest, preferring the study UTM CRS if present and otherwise the largest observation group. Exclude and report incompatible lattices (e.g. overlapping neighboring UTM tiles); do not resample spectral observations onto another lattice. Inspect coverage QA before interpretation. Within a common lattice, different tile origins may differ by integer pixel offsets only. The same UAV aggregation protocol as 02 applies (30×30 sample centers per satellite pixel, ≥90% valid UAV support, 100 m inward ROI, best-supported mosaic). Annual median of jointly valid observations; at least three clear observations (not necessarily unique days). Use the same L30/S30 bands and Fmask as NB08bis 14. Use only complete calendar years before UAV survey. The 2013 archive is incomplete; full available history is compared with a fixed six-year pre-survey window (2014–2019 for Yoko). No interpolation across missing years. Scatterplots and descriptive correlations only; no fitting or satellite-predicted Musanga values. Spectral recovery is not necessarily ecological recovery. References: https://developers.google.com/earth-engine/datasets/catalog/NASA_HLS_HLSL30_v002 and https://developers.google.com/earth-engine/datasets/catalog/NASA_HLS_HLSS30_v002.

In [ ]:
#@title 06A — Verify HLS native grid and create resumable cache
HIDS=['NASA/HLS/HLSL30/v002','NASA/HLS/HLSS30/v002']
HBANDS=['NDVI','NDMI','NBR','NIR','SWIR1','SWIR2']
HRAW=[['B4','B5','B6','B7'],['B4','B8A','B11','B12']]
HSTART=2013;HMINOBS=3;HMINYEARS=5;HCOMMON=6
# Same subpixel support, ROI and native-pixel size in meters as the earlier ecological analysis.
HTILE=64;HSUB=30
def same_hgrid(p,q):
 c,a=p;d,b=q
 # Compare actual parsed definitions, not names or EPSG codes.
 if rasterio.crs.CRS.from_user_input(c)!=rasterio.crs.CRS.from_user_input(d) or not np.allclose([a.a,a.b,a.d,a.e],[b.a,b.b,b.d,b.e],rtol=1e-9,atol=1e-9):return False
 x,y=(~a)*(b.c,b.f)
 return abs(x-round(x))<1e-5 and abs(y-round(y))<1e-5
def native_hgrid(img,band):
 projection=img.select(band).projection()
 p=projection.getInfo();a=Affine(*p['transform'])
 assert abs(a.b)<1e-9 and abs(a.d)<1e-9 and np.isclose(a.a,30) and np.isclose(a.e,-30),p
 # Preserve the original source definition, including unnamed datum WKT.
 # Earth Engine PixelGrid accepts crsWkt as an alternative to crsCode.
 raw=p.get('wkt') or p.get('crs_wkt') or p.get('crs') or projection.crs().getInfo()
 assert raw,('Missing native HLS CRS',band,p)
 try: native=rasterio.crs.CRS.from_user_input(raw)
 except Exception as exc:raise ValueError(('Cannot parse original HLS CRS',band,p)) from exc
 assert native.is_projected and native.linear_units.lower() in ('metre','meter','metres','meters','m'),p
 return raw,a
def hls_crs_grid(c):
 # Do not force an EPSG authority identifier for an unnamed datum.
 if c.upper().startswith('EPSG:'):return {'crsCode':c}
 return {'crsWkt':c if '[' in c else rasterio.crs.CRS.from_user_input(c).to_wkt()}
def prefer_study_zone(c):
 # Only a ranking preference, never a projection replacement.
 from pyproj import CRS as PCRS
 p=PCRS.from_user_input(c)
 zone=p.utm_zone
 if zone is None and p.coordinate_operation is not None:
  params={x.name.lower():x.value for x in p.coordinate_operation.params}
  lon=params.get('longitude of natural origin',params.get('central meridian'))
  north=params.get('false northing')
  return lon is not None and np.isclose(lon,27) and north is not None and np.isclose(north,0)
 return zone=='35N'
def tag_grid(im):
 # All selected optical bands and Fmask must share a projection. Projection() raises if not.
 p=im.projection()
 return im.set({'_grid_crs':p.crs(),'_grid_tf':p.transform()})
HC={};HG={};HREG={};HTILES={};inventory=[]
for s in SITES:
 g=GEOM[s];ll=gpd.GeoSeries([g],crs=CRS).to_crs('EPSG:4326').iloc[0]
 region=ee.Geometry.Rectangle(list(ll.bounds),proj='EPSG:4326',geodesic=False)
 records=[];collections=[]
 for j,asset in enumerate(HIDS):
  col=ee.ImageCollection(asset).filterBounds(region).filterDate(f'{HSTART}-01-01',f'{YEARS[s]}-01-01').select(HRAW[j]+['Fmask']).map(tag_grid)
  collections.append(col)
  # One representative per exact source projection, with all scene counts.
  reps=col.distinct(['_grid_crs','_grid_tf']).sort('system:index')
  nr=reps.size().getInfo()
  if nr==0:continue
  items=reps.toList(nr)
  for k in range(nr):
   im=ee.Image(items.get(k))
   tags=im.toDictionary(['_grid_crs','_grid_tf','system:index']).getInfo()
   p=native_hgrid(im,HRAW[j][0])
   for b in HRAW[j][1:]+['Fmask']:assert same_hgrid(p,native_hgrid(im,b))
   flt=ee.Filter.And(ee.Filter.eq('_grid_crs',tags['_grid_crs']),ee.Filter.eq('_grid_tf',tags['_grid_tf']))
   cnt=col.filter(flt).size().getInfo()
   records.append(dict(sensor=j,proj=p,flt=flt,count=cnt,index=tags['system:index'],crs=p[0],transform=list(p[1])[:6]))
 assert records,f'No historical HLS scenes: {s}'
 groups=[]
 for rec in records:
  found=next((z for z in groups if same_hgrid(z[0]['proj'],rec['proj'])),None)
  if found is None:groups.append([rec])
  else:found.append(rec)
 # Prefer the study's UTM zone if it is an actual source lattice; otherwise use largest native group.
 groups.sort(key=lambda z:(prefer_study_zone(z[0]['crs']),sum(r['count'] for r in z)),reverse=True)
 chosen=groups[0];p=chosen[0]['proj'];HG[s]=p
 for group in groups:
  for rec in group:
   inventory.append(dict(site=s,sensor=HIDS[rec['sensor']],crs=rec['crs'],transform=rec['transform'],representative=rec['index'],n_scenes=rec['count'],retained=group is chosen))
 for j,col in enumerate(collections):
  filters=[rec['flt'] for rec in chosen if rec['sensor']==j]
  if not filters:HC[s,j]=col.limit(0)
  else:HC[s,j]=col.filter(filters[0] if len(filters)==1 else ee.Filter.Or(*filters))
 c,a=p;geom=gpd.GeoSeries([g],crs=CRS).to_crs(c).iloc[0];HREG[s]=geom
 x0,y0,x1,y1=geom.bounds;cl,rt=(~a)*(x0,y1);cr,rb=(~a)*(x1,y0)
 tiles=[]
 for row in range(math.floor(rt/HTILE)*HTILE,math.ceil(rb/HTILE)*HTILE,HTILE):
  for col in range(math.floor(cl/HTILE)*HTILE,math.ceil(cr/HTILE)*HTILE,HTILE):
   l,t=a*(col,row);rr,b=a*(col+HTILE,row+HTILE)
   if geom.intersects(box(l,b,rr,t)):tiles.append((row,col))
 assert tiles;HTILES[s]=tiles
 print(s,c,list(a)[:6],len(tiles),'native tiles;',sum(z['count'] for z in chosen),'retained scenes of',sum(z['count'] for z in records),flush=True)
HCFG=dict(version='native_HLS_ecology_v2_wkt',sources=HIDS,years=YEARS,roi_sha=hashlib.sha256(roi.wkb).hexdigest(),mosaics=meta,model=M['model_sha'],policy=M['policy_sha'],support=SUPPORT,sub=HSUB,tile=HTILE,start=HSTART,min_obs=HMINOBS,min_years=HMINYEARS,common_years=HCOMMON,mask='Fmask bits 0–5 clear, aerosol !=3, joint reflectance mask, positive index denominators',grids={s:[c,list(a)[:6]] for s,(c,a) in HG.items()},inventory=inventory)
HOUT=ROOT/'Outputs/09_ecology_v5'/'06_native_HLS'/digest(HCFG)[:20];HOUT.mkdir(parents=True,exist_ok=True)
HU=HOUT/'uav_tiles';HA=HOUT/'annual_tiles';HU.mkdir(exist_ok=True);HA.mkdir(exist_ok=True)
atomic_json(HOUT/'config.json',HCFG);display(pd.DataFrame(inventory));pd.DataFrame(inventory).to_csv(HOUT/'source_grid_inventory.csv',index=False)
print('06A COMPLETE',HOUT)


In [ ]:
#@title 06B — Native 30 m UAV cover and annual HLS spectra (resumable)
def uav_hls_tile(s,row,col,stack):
 c,a=HG[s];tr=a*Affine.translation(col,row);hi=tr*Affine.scale(1/HSUB,1/HSUB);size=HTILE*HSUB
 inside=rasterize([(HREG[s],1)],out_shape=(size,size),transform=hi,fill=0,dtype='uint8').astype(bool)
 def reduce(z):return z.reshape(HTILE,HSUB,HTILE,HSUB).sum((1,3))
 best=np.zeros((HTILE,HTILE),dtype=np.int32);pos=best.copy();source=np.full(best.shape,-1,dtype=np.int16)
 for i,(name,p,b) in enumerate(MOS[s]):
  with WarpedVRT(stack[name],crs=c,transform=hi,width=size,height=size,resampling=Resampling.nearest,src_nodata=-9999,nodata=-9999,warp_mem_limit=128) as vrt:z=vrt.read(2)
  valid=inside&((z==0)|(z==1));nv=reduce(valid);npix=reduce(valid&(z==1))
  better=nv>best;best[better]=nv[better];pos[better]=npix[better];source[better]=i
 keep=best>=math.ceil(SUPPORT*HSUB*HSUB)
 rr,cc=np.where(keep);xx,yy=tr*(cc+.5,rr+.5)
 return pd.DataFrame(dict(site=s,row=rr+row,col=cc+col,x_native=xx,y_native=yy,survey_year=YEARS[s],support=best[keep]/(HSUB*HSUB),cover_pp=100*pos[keep]/best[keep],mosaic=[MOS[s][i][0] for i in source[keep]]))
def clean_hls(im,j):
 qa=im.select('Fmask')
 ok=qa.bitwiseAnd(63).eq(0).And(qa.rightShift(6).bitwiseAnd(3).neq(3))
 b=im.select(HRAW[j],['R','NIR','SWIR1','SWIR2']).toFloat()
 r,n,s1,s2=[b.select(k) for k in ['R','NIR','SWIR1','SWIR2']]
 def idx(a,b,name):return a.subtract(b).divide(a.add(b)).rename(name)
 valid=ok.And(b.mask().reduce(ee.Reducer.min())).And(n.add(r).gt(0)).And(n.add(s1).gt(0)).And(n.add(s2).gt(0))
 return ee.Image.cat([idx(n,r,'NDVI'),idx(n,s1,'NDMI'),idx(n,s2,'NBR'),n,s1,s2]).updateMask(valid).copyProperties(im,['system:time_start','system:index'])
def annual_hls(s,row,col,year,g):
 c,a=HG[s];tr=a*Affine.translation(col,row);l,t=tr*(0,0);r,b=tr*(HTILE,HTILE)
 area=ee.Geometry.Rectangle([l,b,r,t],proj=c,geodesic=False)
 start=f'{year}-01-01';end=f'{year+1}-01-01'
 c1=HC[s,0].filterBounds(area).filterDate(start,end).map(lambda im:clean_hls(im,0))
 c2=HC[s,1].filterBounds(area).filterDate(start,end).map(lambda im:clean_hls(im,1))
 empty=ee.Image.constant([0]*len(HBANDS)).rename(HBANDS).updateMask(ee.Image.constant(0)).toFloat()
 collection=c1.merge(c2).merge(ee.ImageCollection.fromImages([empty]))
 image=ee.Image.cat([collection.median().select(HBANDS).unmask(-9999,sameFootprint=False),collection.select('NDVI').count().unmask(0,sameFootprint=False).rename('N_CLEAR')]).toFloat()
 grid=dict(dimensions=dict(width=HTILE,height=HTILE),affineTransform=dict(scaleX=tr.a,shearX=tr.b,translateX=tr.c,shearY=tr.d,scaleY=tr.e,translateY=tr.f),**hls_crs_grid(c))
 for attempt in range(5):
  try:arr=ee.data.computePixels(dict(expression=image,fileFormat='NUMPY_NDARRAY',grid=grid));break
  except Exception as e:
   print('RETRY',s,row,col,year,str(e)[:180],flush=True)
   if attempt==4:raise
   time.sleep(2**attempt)
 assert arr.shape==(HTILE,HTILE) and set(arr.dtype.names)==set(HBANDS+['N_CLEAR'])
 rr=g.row.to_numpy(int)-row;cc=g.col.to_numpy(int)-col
 d=g[['site','row','col']].copy();d['year']=year
 count=arr['N_CLEAR'][rr,cc];z=np.column_stack([arr[b][rr,cc] for b in HBANDS])
 valid=(count>=HMINOBS)&np.isfinite(z).all(axis=1)&(z!=-9999).all(axis=1)
 d['n_clear']=count.astype(int);d['valid_year']=valid
 for k,b in enumerate(HBANDS):d[b]=np.where(valid,z[:,k],np.nan)
 return d
uref=[];annual=[]
for s in SITES:
 with ExitStack() as es:
  stack={name:es.enter_context(rasterio.open(p)) for name,p,b in MOS[s]}
  for i,(row,col) in enumerate(HTILES[s]):
   f=HU/f'{s}_{row}_{col}.parquet'
   if f.is_file() and not FORCE_REBUILD:g=pd.read_parquet(f)
   else:g=uav_hls_tile(s,row,col,stack);atomic_parquet(f,g)
   if not len(g):print(s,i+1,'/',len(HTILES[s]),'empty',flush=True);continue
   assert g.site.eq(s).all() and g.support.ge(SUPPORT).all() and g.cover_pp.between(0,100).all()
   uref.append(g)
   for year in range(HSTART,YEARS[s]):
    f=HA/f'{s}_{row}_{col}_{year}.parquet'
    if f.is_file() and not FORCE_REBUILD:d=pd.read_parquet(f)
    else:d=annual_hls(s,row,col,year,g);atomic_parquet(f,d)
    assert d[['site','row','col']].reset_index(drop=True).equals(g[['site','row','col']].reset_index(drop=True)) and d.year.eq(year).all()
    annual.append(d)
   print(s,i+1,'/',len(HTILES[s]),'UAV pixels',len(g),flush=True)
U6=pd.concat(uref,ignore_index=True);A6=pd.concat(annual,ignore_index=True)
K6=['site','row','col']
assert set(U6.site)==set(SITES) and not U6.duplicated(K6).any() and not A6.duplicated(K6+['year']).any()
assert (A6.year<A6.site.map(YEARS)).all()
assert len(A6)==sum((YEARS[s]-HSTART)*len(U6[U6.site.eq(s)]) for s in SITES)
atomic_parquet(HOUT/'HLS_native_UAV_reference.parquet',U6);atomic_parquet(HOUT/'HLS_native_annual.parquet',A6)
q=A6.groupby(K6).valid_year.sum().rename('valid_years').reset_index().merge(U6,on=K6,validate='one_to_one')
qa=q.groupby('site').agg(n_pixels=('cover_pp','size'),mean_UAV_support=('support','mean'),mean_valid_years=('valid_years','mean'),n_ge5_years=('valid_years',lambda z:(z>=HMINYEARS).sum()),n_zero_years=('valid_years',lambda z:(z==0).sum())).reset_index()
qa.to_csv(HOUT/'coverage_QA.csv',index=False);display(qa)
qa2=A6.groupby(['site','year']).agg(n_pixels=('valid_year','size'),n_valid=('valid_year','sum'),median_clear=('n_clear','median')).reset_index()
qa2.to_csv(HOUT/'annual_coverage_QA.csv',index=False);display(qa2)
atomic_json(HOUT/'extraction_complete.json',dict(complete=True,n_pixels=len(U6),n_pixel_years=len(A6)))
print('06B COMPLETE',HOUT)


In [ ]:
#@title 06C — Simple native-pixel scatterplots and fixed-window sensitivity
import warnings
from scipy.stats import spearmanr
assert json.loads((HOUT/'extraction_complete.json').read_text())['complete']
U6=pd.read_parquet(HOUT/'HLS_native_UAV_reference.parquet');A6=pd.read_parquet(HOUT/'HLS_native_annual.parquet')
K6=['site','row','col'];F6=[]
def features6(g,s,window):
 years=np.arange(HSTART if window=='full' else YEARS[s]-HCOMMON,YEARS[s])
 u=U6[U6.site.eq(s)].reset_index(drop=True);idx=pd.MultiIndex.from_frame(u[K6])
 a=g[g.year.isin(years)]
 out=u.copy();out['window']=window;out['n_window_years']=len(years)
 for band in HBANDS:
  z=a.pivot(index=K6,columns='year',values=band).reindex(index=idx,columns=years).to_numpy(float)
  ok=np.isfinite(z);n=ok.sum(1);eligible=n>=HMINYEARS
  with warnings.catch_warnings():
   warnings.simplefilter('ignore',category=RuntimeWarning)
   med=np.nanmedian(z,axis=1);avg=np.nanmean(z,axis=1);sd=np.nanstd(z,axis=1)
  dt=years[None,:]-np.divide((ok*years[None,:]).sum(1),n,out=np.zeros(len(n),float),where=n>0)[:,None]
  den=(ok*dt**2).sum(1);trend=np.divide((dt*np.where(ok,z,0)).sum(1),den,out=np.full(len(z),np.nan),where=den>0)
  # Adjacent valid annual observations only; no interpolation through gaps.
  delta=np.diff(z,axis=1);adj=np.isfinite(delta)
  k=np.argmin(np.where(adj,delta,np.inf),axis=1)
  drop=delta[np.arange(len(z)),k];drop=np.where(adj.any(1),np.maximum(-drop,0),np.nan)
  rec=np.full(len(z),np.nan)
  if band in ['NDVI','NDMI','NBR']:
   # A decline of >=0.05 index units is a descriptive spectral event, not validated disturbance.
   for i in np.flatnonzero(eligible&(drop>=.05)):
    trough=k[i]+1;ix=np.flatnonzero(ok[i]&(np.arange(len(years))>trough))
    if len(ix):rec[i]=(z[i,ix[-1]]-z[i,trough])/(years[ix[-1]]-years[trough])
  out[band+'_valid_years']=n
  for name,v in [('median',med),('mean',avg),('sd',sd),('trend',trend),('largest_decline',drop),('recovery_slope',rec)]:
   if name=='recovery_slope' and band not in ['NDVI','NDMI','NBR']:continue
   out[band+'_'+name]=np.where(eligible,v,np.nan)
 return out
for s in SITES:
 g=A6[A6.site.eq(s)]
 for window in ['full','common6']:F6.append(features6(g,s,window))
F6=pd.concat(F6,ignore_index=True);atomic_parquet(HOUT/'HLS_native_features.parquet',F6)
rows=[]
for window in ['full','common6']:
 for band in HBANDS:
  fields=['median','mean','sd','trend','largest_decline']+(['recovery_slope'] if band in ['NDVI','NDMI','NBR'] else [])
  fig,axs=plt.subplots(len(SITES),len(fields),figsize=(3.2*len(fields),3*len(SITES)),sharey=True,squeeze=False,layout='constrained')
  for i,s in enumerate(SITES):
   g=F6[F6.site.eq(s)&F6.window.eq(window)]
   for j,field in enumerate(fields):
    col=band+'_'+field;ax=axs[i,j];q=g[g[col].notna()]
    x=q[col];y=q.cover_pp
    rho=spearmanr(x,y).statistic if len(q)>2 and x.nunique()>1 and y.nunique()>1 else np.nan
    ax.scatter(x,y,s=3,alpha=.10,rasterized=True)
    ax.set(title=f'{s} | n={len(q):,} | rho={rho:.2f}',xlabel=band+' '+field.replace('_',' '),ylim=(0,100))
    if j==0:ax.set_ylabel('UAV Musanga cover (%)')
    rows.append(dict(window=window,site=s,band=band,statistic=field,n=len(q),n_total=len(g),rho=rho))
  fig.suptitle(f'{band} — {window} historical window')
  fig.savefig(HOUT/f'{band}_{window}_scatter.png',dpi=180);plt.show();plt.close(fig)
stats=pd.DataFrame(rows);stats.to_csv(HOUT/'raw_scatter_summary.csv',index=False)
display(stats.pivot_table(index=['site','band','statistic'],columns='window',values='rho').round(3))
# Compare spectral levels with the same fixed six-year relative window for every forest.
# Means/medians include all valid annual observations; no imputed years, >=5/6 years required.
qa=F6.groupby(['window','site']).agg(n=('cover_pp','size'),eligible=('NBR_median','count'),mean_valid_years=('NBR_valid_years','mean')).reset_index()
qa.to_csv(HOUT/'window_support.csv',index=False);display(qa)
atomic_json(HOUT/'completion.json',dict(complete=True,n_native_pixels=len(U6),n_pixel_years=len(A6),config=HCFG,limitations=['observational spatial chronosequence','unequal full observation windows','spectral event not validated forest disturbance','overlapping incompatible source grids excluded and reported','UAV-derived estimated cover']))
print('06 NATIVE HLS COMPLETE',HOUT)
